<a href="https://colab.research.google.com/github/nonlinearbranch/ml-work/blob/main/transfer_learning_fashion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
 import pandas as pd
 from sklearn.model_selection import train_test_split
 import torch
 from torch.utils.data import Dataset, DataLoader
 import torch.nn as nn
 import torch.optim as optim
 import matplotlib.pyplot as plt
 from torch.nn import Sequential


In [3]:
#for reproducability
torch.manual_seed(42)

In [4]:
#check gpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [5]:
df = pd.read_csv('fashion-mnist_train.csv', on_bad_lines='skip')
df.tail()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
59995,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59996,1,0,0,0,0,0,0,0,0,0,...,73,0,0,0,0,0,0,0,0,0
59997,8,0,0,0,0,0,0,0,0,0,...,160,162,163,135,94,0,0,0,0,0
59998,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59999,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

In [7]:
X_train , X_test , y_train , y_test = train_test_split(X, y, test_size = 0.2, random_state=42)

In [8]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [9]:
X_train_tensor.shape

torch.Size([48000, 784])

In [10]:
#transformations
from torchvision import transforms
custom_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])
])
    
    

In [11]:
from PIL import Image
import numpy as np

class CustomDataset(Dataset):
  def __init__(self, X, y, transform):
    self.X = X.reshape(-1, 1, 28, 28)
    self.y = y
    self.transform = transform

  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    
    #resize to (28.28)
    image = self.X[idx].reshape(28,28)
    
    #change black and white to color
    image = np.stack([image]*3, axis = -1)
        
    #change dtype to np.uint8
    image = image.astype(np.uint8)
    
    #convert array to PIL Image
    image = Image.fromarray(image)
    
    
    #apply transforms
    image = self.transform(image)
    
    
    return image, torch.tensor(self.y[idx], dtype = torch.long)

In [12]:
#dataset and dataloader

train_dataset = CustomDataset(X_train_tensor, y_train_tensor, transform = custom_transform)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor, transform = custom_transform)

#create dataloader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory = True)

In [13]:
# fetch pretrained model

import torchvision.models as models
vgg16 = models.vgg16(pretrained = True)


/home/shira/miniconda3/envs/dl/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/shira/miniconda3/envs/dl/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [14]:
for param in vgg16.features.parameters():
    param.requires_grad = False

In [15]:
vgg16.classifier = nn.Sequential(
    nn.Linear(25088, 1024),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, 10)
)

In [16]:
vgg16 = vgg16.to(device)

In [ ]:
  #train

  #learning parameters
  epochs = 22
  learning_rate = 0.0001

  #optimizer
  optimizer = torch.optim.Adam(vgg16.classifier.parameters(), lr=learning_rate)

  #loss
  loss_fn = nn.CrossEntropyLoss()

  #model train loop( with deciding optimizer and loss fn)

  num_batches = len(train_loader)

  for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:

        #move data to gpu
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        # Forward pass
        y_pred = vgg16(batch_features.float())
        # Compute loss
        loss = loss_fn(y_pred, batch_labels)
        #gradient 0
        optimizer.zero_grad()

        #backward pass
        loss.backward()

        #gradient clipping
        torch.nn.utils.clip_grad_norm_(vgg16.parameters(), max_norm=1.0)

        #parameters update
        optimizer.step()


        total_epoch_loss += loss.item()

    print(f"Epoch: {epoch+1}/{epochs}, Loss: {total_epoch_loss/num_batches}")


  #evaluate model and accuracy

  vgg16.eval()

  with torch.no_grad():
     total = 0
     correct = 0
     #test accuracy evaluation
     for batch_features, batch_labels in test_loader:

          #move data to gpu
          batch_features = batch_features.to(device)
          batch_labels = batch_labels.to(device)

          y_pred = vgg16(batch_features.float())
          # Extract the predicted class indices from the tuple returned by torch.max
          _, predicted_indices = torch.max(y_pred, 1)
          total = total + batch_labels.shape[0]
          # Compare predicted indices with true labels
          correct = correct + (predicted_indices == batch_labels).sum().item()

     test_accuracy = correct / total
     print(f"test accuracy: {test_accuracy}")

     #training accuracy

     for batch_features, batch_labels in train_loader:

          #move data to gpu
          batch_features = batch_features.to(device)
          batch_labels = batch_labels.to(device)

          y_pred = vgg16(batch_features.float())
          # Extract the predicted class indices from the tuple returned by torch.max
          _, predicted_indices = torch.max(y_pred, 1)
          total = total + batch_labels.shape[0]
          # Compare predicted indices with true labels
          correct = correct + (predicted_indices == batch_labels).sum().item()

     train_accuracy = correct / total
     print(f"train accuracy: {train_accuracy}")


/tmp/ipykernel_6056/1432804654.py:32: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return image, torch.tensor(self.y[idx], dtype = torch.long)


Epoch: 1/22, Loss: 0.9358414278576771
Epoch: 2/22, Loss: 0.9269148117204508
Epoch: 3/22, Loss: 0.8245391240703563
Epoch: 4/22, Loss: 0.7369631950656573
Epoch: 5/22, Loss: 0.7527184187943736
Epoch: 6/22, Loss: 0.7429494986931483
Epoch: 7/22, Loss: 0.6904044282610218
Epoch: 8/22, Loss: 0.6612235124011835
Epoch: 9/22, Loss: 0.6236621540933848
Epoch: 10/22, Loss: 0.6090235390613477
Epoch: 11/22, Loss: 0.5890807868431632
Epoch: 12/22, Loss: 0.5900402101402481
Epoch: 13/22, Loss: 0.5715048835376898


In [ ]:
#model class should be initialized priorly

#objective function
def objective_function(trial):
  #take next hyperparameter from search space
  num_hidden_layers = trial.suggest_int('num_hidden_layers', 1, 5)
  num_neurons_per_layer = trial.suggest_int('num_neurons_per_layer', 8, 128, step = 8)
  learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-1, log = True)
  dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5, step = 0.1)
  batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128])
  optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'SGD', 'RMSprop'])
  weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-1, log = True)


  #initialize model
  input_dim = 784
  output_dim = 10
  model = neural_network(input_dim, num_hidden_layers, num_neurons_per_layer, output_dim, dropout_rate)
  model = model.to(device)

  #learning parameters
  epochs = 30

  #optimizer
  if optimizer_name == 'Adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

  elif optimizer_name == 'RMSprop':
    optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

  elif optimizer_name == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

  #loss
  loss_fn = nn.CrossEntropyLoss()

  #model train loop( with deciding optimizer and loss fn)

  num_batches = len(train_loader)

  for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:

        #move data to gpu
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        # Forward pass
        y_pred = model(batch_features.float())
        # Compute loss
        loss = loss_fn(y_pred, batch_labels)
        #gradient 0
        optimizer.zero_grad()

        #backward pass
        loss.backward()

        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        #parameters update
        optimizer.step()

        total_epoch_loss += loss.item()

  #evaluate model and accuracy

  model.eval()

  with torch.no_grad():
     total = 0
     correct = 0
     #test accuracy evaluation
     for batch_features, batch_labels in test_loader:

          #move data to gpu
          batch_features = batch_features.to(device)
          batch_labels = batch_labels.to(device)

          y_pred = model(batch_features.float())
          # Extract the predicted class indices from the tuple returned by torch.max
          _, predicted_indices = torch.max(y_pred, 1)
          total = total + batch_labels.shape[0]
          # Compare predicted indices with true labels
          correct = correct + (predicted_indices == batch_labels).sum().item()
     accuracy = correct / total
  return accuracy





In [ ]:
!pip install optuna

In [ ]:
import optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective_function, n_trials=10)

/home/shira/miniconda3/envs/dl/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-07-05 19:23:13,156] A new study created in memory with name: no-name-94b3f0fe-584b-4c2f-96aa-2c49f0b9c6e7
[W 2026-07-05 19:23:13,159] Trial 0 failed with parameters: {'num_hidden_layers': 2, 'num_neurons_per_layer': 80, 'learning_rate': 0.0014695147516562094, 'dropout_rate': 0.30000000000000004, 'batch_size': 64, 'optimizer': 'Adam', 'weight_decay': 0.014017434930043678} because of the following error: TypeError('neural_network.__init__() takes 2 positional arguments but 6 were given').
Traceback (most recent call last):
  File "/home/shira/miniconda3/envs/dl/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp

TypeError: neural_network.__init__() takes 2 positional arguments but 6 were given

In [ ]:
study.best_value
study.best_params

In [ ]:
model2.eval()

with torch.no_grad():
     total = 0
     correct = 0
     #test accuracy evaluation
     for batch_features, batch_labels in test_loader:

          #move data to gpu
          batch_features = batch_features.to(device)
          batch_labels = batch_labels.to(device)

          y_pred = model2(batch_features.float())
          # Extract the predicted class indices from the tuple returned by torch.max
          _, predicted_indices = torch.max(y_pred, 1)
          total = total + batch_labels.shape[0]
          # Compare predicted indices with true labels
          correct = correct + (predicted_indices == batch_labels).sum().item()

     test_accuracy = correct / total
     print(f"test accuracy: {test_accuracy}")

     #training accuracy

     for batch_features, batch_labels in train_loader:

          #move data to gpu
          batch_features = batch_features.to(device)
          batch_labels = batch_labels.to(device)

          y_pred = model2(batch_features.float())
          # Extract the predicted class indices from the tuple returned by torch.max
          _, predicted_indices = torch.max(y_pred, 1)
          total = total + batch_labels.shape[0]
          # Compare predicted indices with true labels
          correct = correct + (predicted_indices == batch_labels).sum().item()

     train_accuracy = correct / total
     print(f"train accuracy: {train_accuracy}")


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


test accuracy: 0.892
train accuracy: 0.9242166666666667
